## 6.0 Learn how to do RAG to build Document GPT App: 
- RAG: 유저가 제공한 데이터를 LLM에 같이 전달해 그 능력을 강화시킨다.

## 6.1 RAG의 첫 번째 단계인 Retrieval (LangChain Module)
![](https://python.langchain.com/v0.1/assets/images/data_connection-95ff2033a8faa5f3ba41376c0f6dd32a.jpg)

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.document_loaders import PyPDFLoader
from langchain.document_loaders import UnstructuredFileLoader # can load txt, pptx, html, img, and more.
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter # 글을 문단으로 나눈다.

splitter = RecursiveCharacterTextSplitter( 
    chunk_size=200, # 글자 수 200개 단위로 덩어리를 나눠줘
    chunk_overlap=50, # 분할할 때 문장이 잘리지 않게 앞 조각 끝 부분을 가져온다.
) 

splitter = CharacterTextSplitter(
    separator="\n" # \n을 기준으로 나눠줘
    chunk_size=600,
    chunk_overlap=100,
)

loader = TextLoader("./#6_files/document.txt")
loader = PyPDFLoader("./#6_files/document.pdf")
loader = UnstructuredFileLoader("./#6_files/document.docx")

docs = loader.load()
splitter.split_documents(docs)

[Document(page_content='Chapter 3\n\n\'There are three stages in your reintegration,\' said O\'Brien. \'There is\n\nlearning, there is understanding, and there is acceptance. It is time for\n\nyou to enter upon the second stage.\'\n\nAs always, Winston was lying flat on his back. But of late his bonds were\n\nlooser. They still held him to the bed, but he could move his knees a\n\nlittle and could turn his head from side to side and raise his arms from\n\nthe elbow. The dial, also, had grown to be less of a terror. He could\n\nevade its pangs if he was quick-witted enough: it was chiefly when he\n\nshowed stupidity that O\'Brien pulled the lever. Sometimes they got through\n\na whole session without use of the dial. He could not remember how many\n\nsessions there had been. The whole process seemed to stretch out over a\n\nlong, indefinite time--weeks, possibly--and the intervals between the\n\nsessions might sometimes have been days, sometimes only an hour or two.\n\n\'As you lie ther

## 6.2 Tiktoken
- splitter는 default로 chunk_size를 나누는 length의 기준으로, 아래와 같이 python 내장 함수인 len을 사용한다. customize도 가능하다.
- [Tiktoken](https://platform.openai.com/tokenizer)은 model이 텍스트를 세는 방법

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="\n"
    chunk_size=600,
    chunk_overlap=100,
    length_function=len, # this is default value.
)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

## 6.3 Vectors 
- Embedding: 텍스트를 컴퓨터가 이해할 수 있는 숫자로 변환(벡터화)하는 작업. 
- [Vectors](https://turbomaze.github.io/word2vecjson/): 수치화 한 단어 간 연관성을 찾을 수 있도록 저장하는 multi-dimensional 저장소. 
- 벡터 공간에서 질문과 관련 있는 문서들을 찾아와 LLM에 전달한다.
- [참조영상](https://youtu.be/2eWuYf-aZE4?si=zmIq3D4EnntLhhxh)

In [ ]:
# 6.4 Caching Embeddings
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores.chroma import Chroma

cache_dir = LocalFileStore("./#6_files/.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
loader = UnstructuredFileLoader("./#6_files/document.docx")
docs = loader.load_and_split(text_splitter=splitter)
embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = Chroma.from_documents(docs, cached_embeddings)
result = vectorstore.similarity_search("where does Winston live?")

## 6.6 Off-the-shelf Document Chains [Legacy]
1. Stuff: Retrieve한 Documents를 모두 Prompt에 넣는다.
2. Refine: LLM이 Retrieve한 Documents를 하나씩 읽으면서 question에 대한 답변을 생성하고 답변을 개선해 나간다. Doc 수마다 답변을 생성하므로 비용이 비싸다.
3. Map Reduce: Retrieve한 Document별로 각각 체인을 돌린다. 그 결과를 그대로 또는 요약하여 합친 뒤 LLM에 전달해 최종 응답한다.
4. Map Re-rank: 각 Document를 읽고 답변을 한 뒤 점수를 매긴다. 가장 높은 점수를 획득한 답변과 점수를 반환한다.

In [ ]:
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.chains import  RetrievalQA
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores.chroma import Chroma

cache_dir = LocalFileStore("./#6_files/.cache/")

loader = UnstructuredFileLoader("./#6_files")
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
file = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)
vectorstore = Chroma.from_documents(file, cached_embeddings)

llm = ChatOpenAI()
chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # default
    retriever=vectorstore.as_retriever(), # retrieve documents from not only vectorstore, but also database or cloud.
)

chain.run("Where does Winston live?")

## 6.8 Stuff LCEL Chain
- Stuff: Retrieve한 Documents를 모두 Prompt에 넣는다.

- chain 순서
1. Chain.invoke with query
2. Retriever: Retrieve all documents related to query.
3. Prompt: List of Docs go into template's {context} & chain.invoke(query) go into template's human {question} property.

- [Chain Components](https://python.langchain.com/docs/concepts/runnables/):

| Component | Input Type | Output Type |
| --- | --- | --- |
| Prompt | dictionary | PromptValue |
| ChatModel | a string, list of chat messages or a PromptValue | ChatMessage |
| LLM | a string, list of chat messages or a PromptValue | String |
| OutputParser | the output of an LLM or ChatModel | Depends on the parser |
| Retriever | a string | List of Documents |
| Tool | a string or dictionary, depending on the tool | Depends on the tool |

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores.faiss import FAISS
from langchain.schema.runnable import RunnablePassthrough

cache_dir = LocalFileStore("./#6_files/.cache/")

loader = UnstructuredFileLoader("./#6_files/document.docx")
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
file = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)
vectorstore = FAISS.from_documents(file, embeddings)
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(temperature=0.1)
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}",
    ),
    ("human", "{question}"),
])

chain = {"context":retriever, "question":RunnablePassthrough()} | prompt | llm
chain.invoke("Describe Victory Mansions.")

## 6.9 Map Reduce LCEL Chain
- Map Reduce: Retrieve한 Document별로 각각 체인을 돌린다. 그 결과를 그대로 또는 요약하여 합친 뒤 LLM에 전달해 최종 응답한다.
- 장점: Retriever가 반환하는 doc 수가 많으면 stuff에 다 넣을 수 없기 때문에 이 방식이 적합하다.

- chain 순서
1. Retriever -> List[Doc]
2. for doc in docs | prompt: "Read this doc and extract important info to answer the user question." | llm -> len(responses)==len(docs)
3. response 취합 -> one doc
4. final long doc | prompt: "Answer the question with this relavant info" | llm

- [Chain Components](https://python.langchain.com/docs/concepts/runnables/):

| Component | Input Type | Output Type |
| --- | --- | --- |
| Prompt | dictionary | PromptValue |
| ChatModel | a string, list of chat messages or a PromptValue | ChatMessage |
| LLM | a string, list of chat messages or a PromptValue | String |
| OutputParser | the output of an LLM or ChatModel | Depends on the parser |
| Retriever | a string | List of Documents |
| Tool | a string or dictionary, depending on the tool | Depends on the tool |

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.storage import LocalFileStore
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.faiss import FAISS
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

cache_dir=LocalFileStore("./#6_files/.cache/")

loader = UnstructuredFileLoader("./#6_files/document.docx")
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)
file = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)
vectorstore = FAISS.from_documents(file, cached_embeddings)
retriever = vectorstore.as_retriever()

llm = ChatOpenAI()

map_doc_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Use the following portion of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim. If there is no relevant text, return : ''
        -------
        {context}
        """,
    ),
    ("human", "{question}"),
])

#2
map_doc_chain = map_doc_prompt | llm

#2,3
def map_docs(inputs):
    documents = inputs['documents']
    question = inputs['question']
    return ["\n\n".join(
            map_doc_chain.invoke({
            "context":document.page_content,
            "question":question,
            }).content
            ) for document in documents]

#1
map_chain = {"documents": retriever, "question": RunnablePassthrough()} | RunnableLambda(map_docs)

final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Given the following extracted parts of a long document and a question, create a final answer. 
        If you don't know the answer, just say that you don't know. Don't try to make up an answer.
        ------
        {context}
        """,
    ),
    ("human", "{question}"),
])

#4
chain = {"context":map_chain, "question":RunnablePassthrough()} | final_prompt | llm